In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pylab as plt
import scipy.optimize as optim


from src.stategen import sample_haar_generalized, sample_haar_product, StateGenerator
from src.util import sse, sqe

In [ ]:
np.set_printoptions(precision=3, suppress=True)

## Part 1: Study of `min_eta` and `max_eta` parameters

In [ ]:
def bond_entanglement(state):
    L = state.ndim
    res = []
    for i in range(1, L):
        e = sse(state, list(range(i)), batched=False)# / np.log(2)
        res.append(e)
    return np.array(res)

In [ ]:
# psi = sample_haar_generalized(6, 3, 3, eta_min=0.5)
# bond_entanglement(psi)

In [ ]:
xs = np.linspace(0, 12, 100)
lmbda = 10.0
ys = lmbda * np.exp(-lmbda * xs)
plt.plot(xs, ys / np.linalg.norm(ys))
plt.ylim(-0.1, 1.0)

In [ ]:
etas = np.linspace(0.01, 10, 100)
entanglements = []

for eta in etas:
    x = []
    for _ in range(100):
        psi = sample_haar_generalized(6,3,3, eta, eta, permute=False)
        x.append(bond_entanglement(psi)[2])
    entanglements.append(x)

entanglements = np.array(entanglements) / 3

In [ ]:
etas

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))

e_mean = entanglements.mean(1)
e_std  = entanglements.std(1)

ax.plot(etas, e_mean)
ax.fill_between(etas, e_mean - e_std, e_mean + e_std, alpha=0.3)
# ax.set_yscale('log')
# # ax.set_xscale('log')
# ax.set_xlabel("eta")
# ax.set_ylabel("bond entanglement")
# ax.set_title("Bond Entanglement w.r.t $\eta$")
# ax.set_ylim(1e-5, 0)
# ax.axhline(1e-2, color='k')
# ax.axhline(1e-4, color='k')
# ax.set_xticks([0,1,2,3,4,5,6,7,8,9,10])
ax.grid(True)


In [ ]:
def f(eta, L=8, S=4):
    psi = sample_haar_generalized(L, S, S, eta, eta, permute=False)
    return bond_entanglement(psi)[S - 1] / S


xs = np.linspace(0.01, 5, 100)
ys = np.array([f(x, 8) for x in xs])

res = optim.curve_fit(
    lambda x, a1, b1: a1 * x + b1,
    xdata=xs,
    ydata=np.log(ys),
    full_output=False
)

plt.plot(xs, ys)
plt.grid(True)
plt.yscale('log')


coef, covar = res
yhat = np.exp(coef[0] * xs + coef[1])
plt.plot(xs, yhat)
coef

In [ ]:
xs = np.linspace(0.01, 6, 100)
ys = np.array([f(x) for x in xs])

# RRRR-RRRR-RRRR

fig, ax = plt.subplots(figsize=(8,8))
for L in (6,8,10,12):
    ys = np.array([f(x, L, 4) for x in xs])
    ax.plot(xs, ys, label=f"L: {L}")

# Set minor ticks at intervals of 0.5
ax.xaxis.set_minor_locator(mpl.ticker.MultipleLocator(0.25))

ax.grid(True, which="both", linewidth=0.5)
ax.set_yscale('log')
ax.axhline(1e-2, color='k', ls='--')
ax.legend(loc='upper right')
ax.set_xlabel("$\eta$", fontsize=12)
ax.set_ylabel("bond entanglement", fontsize=12)

ax.set_title("Bond Entanglement vs. $\eta$\nSubsystem Size = 4", fontsize=14)

fig.tight_layout()
# fig.savefig("bond-entanglement-vs-eta-2.png", dpi=160)

In [ ]:
# xs = np.linspace(0.01, 5, 100)
# ys = np.array([f(x) for x in xs])

fig, ax = plt.subplots(figsize=(8,8))
# ys = np.array([f(x, 15, 5) for x in xs])
ax.plot(xs, ys, label=f"subsystem size: 5")

# Set minor ticks at intervals of 0.5
ax.xaxis.set_minor_locator(mpl.ticker.MultipleLocator(0.25))

ax.grid(True, which="both", linewidth=0.5)
ax.set_yscale('log')
ax.axhline(1e-2, color='k', ls='--')
ax.legend(loc='upper right')
ax.set_xlabel("$\eta$", fontsize=12)
ax.set_ylabel("bond entanglement", fontsize=12)

ax.set_title("Bond Entanglement vs. $\eta$ for L=15, Subsystem Size = 5", fontsize=14)

fig.tight_layout()
# fig.savefig("bond-entanglement-vs-eta-15q.pdf", dpi=160)

In [ ]:
# N = 100_000

# for _ in range(N):
#     psi = sample_haar_generalized(6, 3, 3, 3.1)

## Part 2: Expected Lengths to Disentangle

In [ ]:
from search import RandomAgent, BeamSearch
from src.quantum_state import VectorQuantumState

In [ ]:
L = 12  # number of qubits
K = 100  # number of search tests (per `eta`)

search_etas = [1.50, 2.00, 2.25, 3.00, 4.00, 5.00, 6.00, 8.00]


In [ ]:
randomS = RandomAgent(epsi=1e-2)
# beamS = BeamSearch(beam_size=200, epsi=1e-2)

In [ ]:
random_search_lengths = {}

for eta in reversed(search_etas):
    print(f"Running Random Search for 4x4x4 product state with eta = {eta:.2f}")
    sgen = StateGenerator(
        sample_haar_generalized,
        L,
        dict(min_subsystem_size=4, max_subsystem_size=4, min_eta=eta, max_eta=eta)
    )
    env = VectorQuantumState(L, 1, "reduced", sgen)

    path_lengths = []
    for _ in range(K):
        env.reset_sub_environment_(0)
        psi = env.states[0]

        random_path = randomS.start(psi, env, verbose=False)
        if random_path is not None:
            path_lengths.append(len(random_path))
            # print(f"\t{len(random_path)}")

    random_search_lengths[eta] = np.array(path_lengths)


In [31]:
list(map(lambda arr: np.mean(arr), random_search_lengths.values()))

[222.77, 249.48, 291.17, 329.07, 1560.33]

In [36]:
random_search_lengths[3.0].mean()

1560.33

In [ ]:
random_search_lengths.keys()

In [ ]:
random_search_lengths = {}

for eta in reversed(search_etas):
    print(f"Running Random Search for 4x4x4 product state with eta = {eta:.2f}")
    sgen = StateGenerator(
        sample_haar_generalized,
        L,
        dict(min_subsystem_size=4, max_subsystem_size=4, min_eta=eta, max_eta=eta)
    )
    env = VectorQuantumState(L, 1, "reduced", sgen)

    path_lengths = []
    for _ in range(K):
        env.reset_sub_environment_(0)
        psi = env.states[0]

        random_path = randomS.start(psi, env, verbose=False)
        if random_path is not None:
            path_lengths.append(len(random_path))
            print(f"\t{len(random_path)}")

    random_search_lengths[eta] = np.array(path_lengths)


Running Beam Search for 4x4x4 product state with eta = 2.25
	61
	57

Running Beam Search for 4x4x4 product state with eta = 2.00
	53
	119

Running Beam Search for 4x4x4 product state with eta = 1.50
	585
	348

Running Beam Search for 4x4x4 product state with eta = 1.25

Running Random Search for 4x4x4 product state with eta = 8.00
	103
	195
Running Random Search for 4x4x4 product state with eta = 6.00
	241
	167
Running Random Search for 4x4x4 product state with eta = 5.00
	196
	160
Running Random Search for 4x4x4 product state with eta = 4.00
	338
	248
Running Random Search for 4x4x4 product state with eta = 3.00
	154
	5359
Running Random Search for 4x4x4 product state with eta = 2.25
	3396
	2746
Running Random Search for 4x4x4 product state with eta = 2.00
	4668
	2165
Running Random Search for 4x4x4 product state with eta = 1.50
	5049
	6086